# Gold | 2025 charger ranking plus updated historical facts

The decision mart remains based on the complete 2025 calendar year. Additional 2026 SCB and electricity observations appear in granular facts; partial 2026 electricity is excluded from the annual price fact to avoid presenting a partial year as a full-year average. Run after silver_databricks_2026.ipynb.


In [0]:
import csv
import json
from pathlib import Path
from datetime import date

from pyspark.sql import Window
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, StructField, StructType

CATALOG = "db_labb2"
BRONZE = f"{CATALOG}.bronze"
SILVER = f"{CATALOG}.silver"
GOLD = f"{CATALOG}.gold"
DECISION_YEAR = 2025
PREVIOUS_YEAR = 2024
spark.conf.set("spark.sql.session.timeZone", "UTC")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD}")

population = spark.table(f"{SILVER}.population")
registrations = spark.table(f"{SILVER}.registrations")
vehicles = spark.table(f"{SILVER}.vehicles_in_use")
prices = spark.table(f"{SILVER}.electricity_prices")

for name, df, minimum in [
    ("population", population, 870),
    ("registrations", registrations, 20880),
    ("vehicles_in_use", vehicles, 870),
    ("electricity_prices", prices, 61548),
]:
    actual = df.count()
    if actual < minimum:
        raise ValueError(f"{SILVER}.{name}: expected at least {minimum} rows, got {actual}")
    print(f"{SILVER}.{name}: {actual} rows")


In [0]:
def bronze_labels(table_id, dimension_id):
    """Read category labels from the stored SCB JSON-stat Bronze responses."""
    result = {}
    raw = spark.table(f"{BRONZE}.scb_{table_id.lower()}").select("payload_json").collect()
    if len(raw) < 12:
        raise ValueError(f"{table_id}: fewer than 12 Bronze responses")
    for response in raw:
        payload = json.loads(response.payload_json)
        labels = payload["dimension"][dimension_id]["category"]["label"]
        for code, label in labels.items():
            if code in result and result[code] != label:
                raise ValueError(f"{table_id}: conflicting {dimension_id} label for {code}")
            result[code] = label
    return result


region_labels = bronze_labels("TAB638", "Region")
fuel_labels = bronze_labels("TAB3277", "Drivmedel")

# The reference CSV is versioned beside this notebook in the Git folder.
# Use labels from the actual Bronze SCB responses to attach municipality codes.
area_file = Path.cwd() / "kommun_price_area_2024.csv"
if not area_file.is_file():
    raise FileNotFoundError(f"Place kommun_price_area_2024.csv beside this notebook: {area_file}")
with area_file.open(encoding="utf-8-sig", newline="") as handle:
    area_rows = list(csv.DictReader(handle))
if (len(area_rows) != 290 or
        {row["price_area"] for row in area_rows} != {"SE1", "SE2", "SE3", "SE4"} or
        len({row["kommun_name"] for row in area_rows}) != 290):
    raise ValueError("Municipality-to-electricity-area reference CSV is incomplete")
area_by_name = {row["kommun_name"]: row["price_area"] for row in area_rows}

population_codes = {row.kommun_code for row in population.select("kommun_code").distinct().collect()}
if len(population_codes) != 290 or set(region_labels) != population_codes:
    raise ValueError("Bronze population labels do not match Silver municipalities")
if len({region_labels[code].strip() for code in population_codes}) != 290:
    raise ValueError("SCB municipality labels are not unique")
unmapped = sorted({region_labels[code].strip() for code in population_codes} - set(area_by_name))
extra = sorted(set(area_by_name) - {region_labels[code].strip() for code in population_codes})
if unmapped or extra:
    raise ValueError(f"Price area mapping does not match SCB labels. Missing: {unmapped}; extra: {extra}")

dim_kommun = spark.createDataFrame(
    [(code, region_labels[code].strip(), area_by_name[region_labels[code].strip()])
     for code in sorted(population_codes)],
    StructType([
        StructField("kommun_code", StringType(), False),
        StructField("kommun_name", StringType(), False),
        StructField("price_area_code", StringType(), False),
    ]),
)
if set(fuel_labels) != {"120", "140"}:
    raise ValueError(f"Unexpected fuel labels in Bronze: {fuel_labels}")
dim_fuel = spark.createDataFrame(
    [(code, fuel_labels[code]) for code in ("120", "140")],
    ["fuel_code", "fuel_name"],
)
dim_year = spark.createDataFrame(
    [(year,) for year in sorted(
        {r.year for r in population.select("year").distinct().collect()} |
        {r.year for r in registrations.select(F.year("month").alias("year")).distinct().collect()} |
        {r.year for r in prices.select(F.year("price_date").alias("year")).distinct().collect()})],
    ["year"])
dim_price_area = spark.createDataFrame(
    [(area,) for area in ("SE1", "SE2", "SE3", "SE4")], ["price_area"]
)
print("Dimensions: 290 municipalities, 2 fuels, dynamic years, 4 price areas")


In [0]:
# Preserve the granular fact tables from the original local Gold notebook.
fact_registrations = registrations.select(
    "kommun_code", "month", "fuel_code", "new_registrations"
)
fact_population = population.select("kommun_code", "year", "population")
fact_vehicle_stock = vehicles.select(
    "kommun_code", "year", "passenger_cars_in_use"
)
fact_electricity_price = prices.select(
    "price_area", "price_date", "time_start", "time_end",
    "sek_per_kwh", "duration_minutes"
)

for name, df, keys in [
    ("fact_registrations", fact_registrations, ["kommun_code", "month", "fuel_code"]),
    ("fact_population", fact_population, ["kommun_code", "year"]),
    ("fact_vehicle_stock", fact_vehicle_stock, ["kommun_code", "year"]),
    ("fact_electricity_price", fact_electricity_price, ["price_area", "time_start"]),
]:
    if df.groupBy(*keys).count().where(F.col("count") > 1).limit(1).count():
        raise ValueError(f"{name}: duplicate business key")

# Weight by the actual duration; 2025 has hourly and quarter-hourly prices.
fact_annual_electricity_price = (
    fact_electricity_price.where(F.year("price_date") == DECISION_YEAR)
    .withColumn("year", F.year("price_date"))
    .groupBy("price_area", "year")
    .agg(
        (F.sum(F.col("sek_per_kwh") * F.col("duration_minutes")) /
         F.sum("duration_minutes")).alias("avg_sek_per_kwh"),
        F.sum("duration_minutes").alias("total_minutes"),
        F.count("*").alias("intervals"),
    )
)
annual_checks = fact_annual_electricity_price.collect()
if (len(annual_checks) != 4 or {row.price_area for row in annual_checks} !=
        {"SE1", "SE2", "SE3", "SE4"} or
        any(row.year != 2025 or row.total_minutes != 365 * 24 * 60 or
            row.intervals != 15387 or row.avg_sek_per_kwh is None
            for row in annual_checks)):
    raise ValueError(f"Annual electricity coverage is incomplete: {annual_checks}")
display(fact_annual_electricity_price.orderBy("price_area"))


In [0]:
# Annual new rechargeable registrations by municipality and fuel code.
annual_registration = (
    fact_registrations
    .withColumn("year", F.year("month"))
    .groupBy("kommun_code", "year")
    .pivot("fuel_code", ["120", "140"])
    .agg(F.sum("new_registrations"))
    .withColumnRenamed("120", "new_battery_electric")
    .withColumnRenamed("140", "new_plug_in_hybrid")
    .withColumn("new_rechargeable",
                F.col("new_battery_electric") + F.col("new_plug_in_hybrid"))
)
if (annual_registration.where(F.col("year").isin(PREVIOUS_YEAR, DECISION_YEAR)).count() != 580 or
        annual_registration.where(
            F.col("new_battery_electric").isNull() |
            F.col("new_plug_in_hybrid").isNull()).limit(1).count()):
    raise ValueError("Annual registrations: incomplete municipality/year/fuel coverage")

prior = (annual_registration.where(F.col("year") == PREVIOUS_YEAR)
         .select("kommun_code", F.col("new_rechargeable").alias("new_rechargeable_2024")))
current = (annual_registration.where(F.col("year") == DECISION_YEAR)
           .select("kommun_code", "new_battery_electric", "new_plug_in_hybrid",
                   F.col("new_rechargeable").alias("new_rechargeable_2025")))

mart_base = (
    dim_kommun
    .join(prior, "kommun_code", "inner")
    .join(current, "kommun_code", "inner")
    .join(fact_population.where(F.col("year") == DECISION_YEAR)
          .select("kommun_code", "population"), "kommun_code", "inner")
    .join(fact_vehicle_stock.where(F.col("year") == DECISION_YEAR)
          .select("kommun_code", "passenger_cars_in_use"), "kommun_code", "inner")
    .join(fact_annual_electricity_price.where(F.col("year") == DECISION_YEAR)
          .select(F.col("price_area").alias("price_area_code"), "avg_sek_per_kwh"),
          "price_area_code", "inner")
    .withColumn("growth_rate_2025",
                F.when(F.col("new_rechargeable_2024") > 0,
                       (F.col("new_rechargeable_2025") -
                        F.col("new_rechargeable_2024")) /
                       F.col("new_rechargeable_2024")))
    .withColumn("new_rechargeable_per_1000_people",
                F.col("new_rechargeable_2025") / F.col("population") * 1000)
    .withColumn("new_rechargeable_per_1000_cars",
                F.col("new_rechargeable_2025") /
                F.col("passenger_cars_in_use") * 1000)
)
if (mart_base.count() != 290 or
        mart_base.select("kommun_code").distinct().count() != 290 or
        mart_base.where((F.col("population") <= 0) |
                        (F.col("passenger_cars_in_use") <= 0)).limit(1).count()):
    raise ValueError("Decision mart: incomplete municipality coverage or invalid denominator")


In [0]:
def add_pct_rank(frame, column, output, ascending=True):
    """Add pandas-style average percentile ranks, including the treatment of ties."""
    direction = F.col(column).asc() if ascending else F.col(column).desc()
    ordering = Window.orderBy(direction)
    ties = Window.partitionBy(column)
    number = frame.count()
    return (frame
            .withColumn("_rank_min", F.rank().over(ordering))
            .withColumn("_tie_count", F.count("*").over(ties))
            .withColumn(output,
                        (F.col("_rank_min") + (F.col("_tie_count") - 1) / 2) /
                        F.lit(number))
            .drop("_rank_min", "_tie_count"))


scored = add_pct_rank(mart_base, "new_rechargeable_2025", "volume_score")
scored = add_pct_rank(scored, "avg_sek_per_kwh", "price_score", ascending=False)

# Small 2024 bases (< 20 registrations) receive a neutral growth score.
stable = (scored.where(F.col("new_rechargeable_2024") >= 20)
          .withColumn("growth_capped", F.greatest(F.lit(-0.5), F.least(
              F.col("growth_rate_2025"), F.lit(1.0)))))
growth_scores = add_pct_rank(stable, "growth_capped", "growth_score")
scored = (scored.join(growth_scores.select("kommun_code", "growth_score"),
                     "kommun_code", "left")
          .fillna({"growth_score": 0.5})
          .withColumn("priority_score", F.lit(0.45) * F.col("volume_score") +
                      F.lit(0.35) * F.col("growth_score") +
                      F.lit(0.20) * F.col("price_score"))
          .withColumn("priority_rank", F.row_number().over(
              Window.orderBy(F.col("priority_score").desc(),
                             F.col("kommun_code").asc())))
          .select("priority_rank", "kommun_code", "kommun_name", "price_area_code",
                  "new_battery_electric", "new_plug_in_hybrid",
                  "new_rechargeable_2024", "new_rechargeable_2025",
                  "growth_rate_2025", "population", "passenger_cars_in_use",
                  "new_rechargeable_per_1000_people", "new_rechargeable_per_1000_cars",
                  "avg_sek_per_kwh", "volume_score", "growth_score", "price_score",
                  "priority_score"))
mart_charger_priority = scored
if (mart_charger_priority.count() != 290 or
        mart_charger_priority.where(F.col("priority_score").isNull()).limit(1).count() or
        mart_charger_priority.select("priority_rank").distinct().count() != 290):
    raise ValueError("Priority score/rank is incomplete")

print("Top 50 price areas:")
display(mart_charger_priority.where(F.col("priority_rank") <= 50)
        .groupBy("price_area_code").count().orderBy("price_area_code"))
display(mart_charger_priority.orderBy("priority_rank").limit(50))


In [0]:
# Inspect the original notebook's example municipalities and test score sensitivity.
display(mart_charger_priority.where(
    F.col("kommun_name").isin("Täby", "Jönköping", "Stockholm")
).orderBy("priority_rank"))

alternative = mart_charger_priority.withColumn(
    "alternative_score",
    F.lit(0.60) * F.col("volume_score") +
    F.lit(0.20) * F.col("growth_score") +
    F.lit(0.20) * F.col("price_score"),
)
alternative_top50 = {
    row.kommun_code for row in alternative.orderBy(
        F.col("alternative_score").desc(), F.col("kommun_code").asc()
    ).limit(50).select("kommun_code").collect()
}
original_top50 = {
    row.kommun_code for row in mart_charger_priority.where(
        F.col("priority_rank") <= 50).select("kommun_code").collect()
}
print("Top 50 overlap under alternative weights:",
      len(original_top50 & alternative_top50), "/ 50")


In [0]:
# Rebuild managed Delta tables on each rerun. All checks above finish before writing.
gold_tables = {
    "dim_kommun": dim_kommun,
    "dim_fuel": dim_fuel,
    "dim_year": dim_year,
    "dim_price_area": dim_price_area,
    "fact_registrations": fact_registrations,
    "fact_population": fact_population,
    "fact_vehicle_stock": fact_vehicle_stock,
    "fact_electricity_price": fact_electricity_price,
    "fact_annual_electricity_price": fact_annual_electricity_price,
    "mart_charger_priority": mart_charger_priority,
}

# Validate each output against its input; the counts grow as new periods arrive.
expected_rows = {
    "dim_kommun": 290, "dim_fuel": 2, "dim_year": dim_year.count(),
    "dim_price_area": 4, "fact_registrations": registrations.count(),
    "fact_population": population.count(),
    "fact_vehicle_stock": vehicles.count(),
    "fact_electricity_price": prices.count(),
    "fact_annual_electricity_price": 4, "mart_charger_priority": 290,
}
for name, df in gold_tables.items():
    target = f"{GOLD}.{name}"
    (df.write.format("delta").mode("overwrite")
       .option("overwriteSchema", "true").saveAsTable(target))
    actual = spark.table(target).count()
    if actual != expected_rows[name]:
        raise ValueError(f"{target}: expected {expected_rows[name]}, found {actual}")
    print(target, "→", actual, "rows")
